In [1]:
from bs4 import BeautifulSoup as soup
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

In [7]:
!pip install undetected_chromedriver

In [6]:
pip install langdetect

In [4]:
pip install pandas google-search-results scikit-learn langdetect

In [9]:
import os
import random
import time
import urllib.parse
from datetime import datetime, timezone
import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langdetect import detect, LangDetectException

# ==========================================
# --- 1. NLP 清洗與去重模組 ---
# ==========================================
def filter_english_only(news_list):
    if not news_list: return []
    english_news = []
    for news in news_list:
        text = news.get("Content", "")
        if not text.strip(): continue
        try:
            if detect(text) == 'en': english_news.append(news)
        except LangDetectException:
            continue
    return english_news

def deduplicate_news(news_list, threshold=0.85):
    if len(news_list) < 2: return news_list
    contents = [n["Content"] for n in news_list]
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(contents)
    cosine_sim = cosine_similarity(tfidf_matrix)
    
    unique_indices = []
    dropped = set()
    for i in range(len(news_list)):
        if i in dropped: continue
        unique_indices.append(i)
        for j in range(i + 1, len(news_list)):
            if cosine_sim[i, j] >= threshold:
                dropped.add(j)
    return [news_list[i] for i in unique_indices]

# ==========================================
# --- 2. 終極擬真抗封鎖抓取邏輯 ---
# ==========================================
def scrape_year_batch(players_dict): 
    all_final_data = []
    
    options = uc.ChromeOptions()
    # 建立全新的乾淨 Profile，用來儲存你手動解開的 Cookie
    #profile_path = os.path.abspath("./ultimate_year_profile")
    #options.add_argument(f"--user-data-dir={profile_path}")
    
    # 🛡️ 硬體與環境偽裝
    options.add_argument("--window-size=1280,800") 
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-notifications")
    options.add_argument("--lang=en-US,en;q=0.9")
    options.add_argument("--start-maximized") # 強制讓視窗跳出來並最大化
    
    # 🌟 精準鎖定 Chrome 149 版
    driver = uc.Chrome(options=options, version_main=149)
    
    # 注入底層 JS 隱藏 webdriver 特徵
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined});"
    })
    
    # 計算進度用
    total_players = len(players_dict)
    current_count = 0
    
    for name, year in players_dict.items():
        current_count += 1
        start_dt = datetime(year - 1, 10, 1, tzinfo=timezone.utc)
        end_dt = datetime(year + 1, 6, 30, tzinfo=timezone.utc)
        
        date_param = f"cdr:1,cd_min:{start_dt.strftime('%m/%d/%Y')},cd_max:{end_dt.strftime('%m/%d/%Y')}"
        query = urllib.parse.quote_plus(name)
        
        temp_player_news = [] 
        page = 0
        max_safe_pages = 10 # 每個球員最多翻10頁(約100篇)，防止無限迴圈
        
        print(f"\n[{current_count}/{total_players}] 🔎 正在檢索: {name} (區間: {start_dt.date()} ~ {end_dt.date()})")
        
        # 🛡️ 擬真行為：從 Google 首頁開始 (Session 預熱)
        try:
            driver.get("https://www.google.com")
            time.sleep(random.uniform(1.5, 3.0))
            search_box = driver.find_element(By.NAME, "q")
            search_box.send_keys(name) 
            time.sleep(random.uniform(0.5, 1.2)) 
            search_box.send_keys(Keys.RETURN) 
            time.sleep(random.uniform(2.0, 4.0))
        except Exception:
            pass 
            
        while page < max_safe_pages:
            start_offset = page * 10
            news_url = f"https://www.google.com/search?q={query}&tbm=nws&tbs={date_param}&start={start_offset}"
            
            print(f"   ➡️ 掃描第 {page + 1} 頁...")
            driver.get(news_url)
            time.sleep(random.uniform(3.5, 6.0)) 

            # 🛡️ 擬真行為：滑鼠軌跡與滾動
            try:
                scroll_depth = random.randint(400, 900)
                driver.execute_script(f"window.scrollBy(0, {scroll_depth});")
                time.sleep(random.uniform(1.0, 2.5))
                
                links = driver.find_elements(By.TAG_NAME, "a")
                if len(links) > 10:
                    random_link = random.choice(links[5:15]) 
                    ActionChains(driver).move_to_element(random_link).perform()
            except Exception:
                pass 

            soup = BeautifulSoup(driver.page_source, "html.parser")
            result_containers = soup.select('div[data-ved]') 
            
            # 🚨 驗證碼人工介入暫停系統！
            if not result_containers:
                page_text = driver.page_source.lower()
                if "recaptcha" in page_text or "unusual traffic" in page_text or "sorry" in driver.title.lower():
                    print("\n" + "⚠️"*20)
                    print("🚨 警告：觸發 Google 驗證碼！")
                    print("⚠️"*20)
                    input("👉 請前往彈出的 Chrome 手動解開驗證碼，看到新聞後，回來終端機按 [Enter] 繼續...")
                    
                    # 你按 Enter 後，重新抓取網頁原始碼
                    soup = BeautifulSoup(driver.page_source, "html.parser")
                    result_containers = soup.select('div[data-ved]') 
                
                # 解鎖後若還是空的，代表該時間區間真的沒新聞了
                if not result_containers:
                    print(f"   🛑 時間區間耗盡！已無新資料。")
                    break
            
            page_item_count = 0
            for res in result_containers:
                title_el = res.find('div', role='heading')
                snippet_el = res.find('div', style=lambda x: x and 'line-clamp' in x)
                
                if title_el and snippet_el:
                    temp_player_news.append({
                        "Player": name,
                        "Draft_Year": year,
                        "Date": "Historical", 
                        "Content": f"{title_el.text} | {snippet_el.text}"
                    })
                    page_item_count += 1
            
            if page_item_count == 0:
                break
                
            page += 1 
            
        print(f"📊 {name}: 總共翻了 {page} 頁，初步收集 {len(temp_player_news)} 筆")
        
        en_news = filter_english_only(temp_player_news)
        clean_news = deduplicate_news(en_news, threshold=0.85)
        
        print(f"✅ {name} 處理完畢，保留 {len(clean_news)} 筆精華數據。")
        all_final_data.extend(clean_news)
        
        # 換球員的超長隨機休息 (極度重要！避免被算總量)
        rest_time = random.uniform(5.5, 9.5)
        print(f"   ☕ 休息 {round(rest_time, 1)} 秒後處理下一位...\n")
        time.sleep(rest_time)
        
    driver.quit()
    return all_final_data

# ==========================================
# --- 3. 讀取 CSV 並執行批次抓取 ---
# ==========================================
if __name__ == "__main__":
    
    # ⚠️ 1. 設定你的原始資料庫檔名與欄位
    CSV_FILE_PATH = "NBA_Normalized_Final.csv" 
    COLUMN_PLAYER = "Player"      
    COLUMN_YEAR = "year_start"
    DraftRank= "DraftRank"
    
    # ⚠️ 2. 設定你要跑的選秀年份 (一次跑一年，穩紮穩打)
    TARGET_YEAR = 2023
    
    print(f"📂 正在讀取總表：{CSV_FILE_PATH}")
    
    try:
        df_all = pd.read_csv(CSV_FILE_PATH)
        
        # 清理年份為空值的資料，並轉為整數 (防止 float 報錯)
        df_all = df_all.dropna(subset=[COLUMN_YEAR])
        df_all[COLUMN_YEAR] = df_all[COLUMN_YEAR].astype(int)
        
        # 篩選出目標年份的球員
        df_target_year = df_all[(df_all[COLUMN_YEAR] == TARGET_YEAR)&(df_all[DraftRank]<31)]
        
        if df_target_year.empty:
            print(f"⚠️ 在檔案中找不到 {TARGET_YEAR} 年的球員資料，請檢查年份是否正確。")
            exit()
            
        # 建立要送給爬蟲的字典 { "球員名": 年份 }
        players_to_run = {}
        for index, row in df_target_year.iterrows():
            players_to_run[row[COLUMN_PLAYER]] = row[COLUMN_YEAR]
            
        print(f"🎯 鎖定 {TARGET_YEAR} 年梯次，共有 {len(players_to_run)} 名球員準備進行抓取！")
        print("💡 提示：請留意終端機畫面，若出現紅綠燈驗證碼，需手動解除。\n")
        
    except FileNotFoundError:
        print(f"❌ 找不到檔案 '{CSV_FILE_PATH}'。請確認檔案與程式碼放在同一個資料夾。")
        exit()
    except KeyError as e:
        print(f"❌ 找不到欄位 {e}。請確認 COLUMN_PLAYER 與 COLUMN_YEAR 名稱正確。")
        exit()

    # 啟動主爬蟲
    final_dataset = scrape_year_batch(players_to_run)
    df_result = pd.DataFrame(final_dataset)
    
    print("\n" + "="*50)
    print(f"🎉 {TARGET_YEAR} 年梯次全數處理完成！總共獲得 {len(final_dataset)} 筆純淨數據。")
    print("="*50)
    
    if not df_result.empty:
        # 自動以年份命名存檔
        output_name = f"NBA_News_{TARGET_YEAR}_Full_Cohort.csv"
        df_result.to_csv(output_name, index=False, encoding='utf-8-sig')
        print(f"\n📁 資料已成功儲存至: {output_name}")
    else:
        print("⚠️ 爬蟲沒有抓到任何有效資料。")

📂 正在讀取總表：NBA_Normalized_Final.csv
🎯 鎖定 2023 年梯次，共有 29 名球員準備進行抓取！
💡 提示：請留意終端機畫面，若出現紅綠燈驗證碼，需手動解除。


[1/29] 🔎 正在檢索: Ochai Agbaji (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
📊 Ochai Agbaji: 總共翻了 1 頁，初步收集 3 筆
✅ Ochai Agbaji 處理完畢，保留 0 筆精華數據。
   ☕ 休息 6.8 秒後處理下一位...


[2/29] 🔎 正在檢索: Paolo Banchero (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
   ➡️ 掃描第 3 頁...
   ➡️ 掃描第 4 頁...

⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️
🚨 警告：觸發 Google 驗證碼！
⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️


👉 請前往彈出的 Chrome 手動解開驗證碼，看到新聞後，回來終端機按 [Enter] 繼續... 


   ➡️ 掃描第 5 頁...
   ➡️ 掃描第 6 頁...
   ➡️ 掃描第 7 頁...
   ➡️ 掃描第 8 頁...
   ➡️ 掃描第 9 頁...
   ➡️ 掃描第 10 頁...
📊 Paolo Banchero: 總共翻了 10 頁，初步收集 120 筆
✅ Paolo Banchero 處理完畢，保留 89 筆精華數據。
   ☕ 休息 8.4 秒後處理下一位...


[3/29] 🔎 正在檢索: Christian Braun (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
📊 Christian Braun: 總共翻了 1 頁，初步收集 7 筆
✅ Christian Braun 處理完畢，保留 1 筆精華數據。
   ☕ 休息 6.6 秒後處理下一位...


[4/29] 🔎 正在檢索: Dyson Daniels (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
   ➡️ 掃描第 3 頁...

⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️
🚨 警告：觸發 Google 驗證碼！
⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️


👉 請前往彈出的 Chrome 手動解開驗證碼，看到新聞後，回來終端機按 [Enter] 繼續... 


   ➡️ 掃描第 4 頁...
   ➡️ 掃描第 5 頁...
   ➡️ 掃描第 6 頁...
   ➡️ 掃描第 7 頁...
   ➡️ 掃描第 8 頁...
   ➡️ 掃描第 9 頁...
   ➡️ 掃描第 10 頁...
📊 Dyson Daniels: 總共翻了 9 頁，初步收集 107 筆
✅ Dyson Daniels 處理完畢，保留 80 筆精華數據。
   ☕ 休息 6.0 秒後處理下一位...


[5/29] 🔎 正在檢索: Ousmane Dieng (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
📊 Ousmane Dieng: 總共翻了 1 頁，初步收集 3 筆
✅ Ousmane Dieng 處理完畢，保留 0 筆精華數據。
   ☕ 休息 8.5 秒後處理下一位...


[6/29] 🔎 正在檢索: Jalen Duren (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
   ➡️ 掃描第 3 頁...

⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️
🚨 警告：觸發 Google 驗證碼！
⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️


...

   ➡️ 掃描第 4 頁...
   ➡️ 掃描第 5 頁...
   ➡️ 掃描第 6 頁...
📊 Dalen Terry: 總共翻了 5 頁，初步收集 54 筆
✅ Dalen Terry 處理完畢，保留 43 筆精華數據。
   ☕ 休息 6.8 秒後處理下一位...


[22/29] 🔎 正在檢索: AJ Griffin (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
📊 AJ Griffin: 總共翻了 1 頁，初步收集 10 筆
✅ AJ Griffin 處理完畢，保留 0 筆精華數據。
   ☕ 休息 5.5 秒後處理下一位...


[23/29] 🔎 正在檢索: Wendell Moore Jr. (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
   ➡️ 掃描第 3 頁...
   ➡️ 掃描第 4 頁...
   ➡️ 掃描第 5 頁...
   ➡️ 掃描第 6 頁...
📊 Wendell Moore Jr.: 總共翻了 5 頁，初步收集 54 筆
✅ Wendell Moore Jr. 處理完畢，保留 42 筆精華數據。
   ☕ 休息 6.9 秒後處理下一位...


[24/29] 🔎 正在檢索: Johnny Davis (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...

⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️
🚨 警告：觸發 Google 驗證碼！
⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️


👉 請前往彈出的 Chrome 手動解開驗證碼，看到新聞後，回來終端機按 [Enter] 繼續... 


   ➡️ 掃描第 2 頁...
📊 Johnny Davis: 總共翻了 1 頁，初步收集 5 筆
✅ Johnny Davis 處理完畢，保留 0 筆精華數據。
   ☕ 休息 7.7 秒後處理下一位...


[25/29] 🔎 正在檢索: Nikola Jović (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
📊 Nikola Jović: 總共翻了 0 頁，初步收集 0 筆
✅ Nikola Jović 處理完畢，保留 0 筆精華數據。
   ☕ 休息 8.9 秒後處理下一位...


[26/29] 🔎 正在檢索: Blake Wesley (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
📊 Blake Wesley: 總共翻了 1 頁，初步收集 6 筆
✅ Blake Wesley 處理完畢，保留 0 筆精華數據。
   ☕ 休息 8.4 秒後處理下一位...


[27/29] 🔎 正在檢索: Patrick Baldwin Jr. (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
   ➡️ 掃描第 3 頁...
   ➡️ 掃描第 4 頁...
   ➡️ 掃描第 5 頁...
   ➡️ 掃描第 6 頁...
📊 Patrick Baldwin Jr.: 總共翻了 5 頁，初步收集 54 筆
✅ Patrick Baldwin Jr. 處理完畢，保留 42 筆精華數據。
   ☕ 休息 7.1 秒後處理下一位...


[28/29] 🔎 正在檢索: Malaki Branham (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...

⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️
🚨 警告：觸發 Google 驗證碼！
⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️


👉 請前往彈出的 Chrome 手動解開驗證碼，看到新聞後，回來終端機按 [Enter] 繼續... 


   ➡️ 掃描第 2 頁...
📊 Malaki Branham: 總共翻了 1 頁，初步收集 7 筆
✅ Malaki Branham 處理完畢，保留 0 筆精華數據。
   ☕ 休息 9.4 秒後處理下一位...


[29/29] 🔎 正在檢索: TyTy Washington Jr. (區間: 2022-10-01 ~ 2024-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
   ➡️ 掃描第 3 頁...
   ➡️ 掃描第 4 頁...
   ➡️ 掃描第 5 頁...
📊 TyTy Washington Jr.: 總共翻了 4 頁，初步收集 39 筆
✅ TyTy Washington Jr. 處理完畢，保留 28 筆精華數據。
   ☕ 休息 7.6 秒後處理下一位...


🎉 2023 年梯次全數處理完成！總共獲得 1275 筆純淨數據。

📁 資料已成功儲存至: NBA_News_2023_Full_Cohort.csv


In [2]:
import os
import shutil
import undetected_chromedriver as uc
import time

print("🧹 步驟一：正在尋找並清除被污染的驅動程式快取...")

# 1. 自動找出隱藏的快取資料夾並刪除 (支援 Windows / Mac)
cache_cleared = False
# Windows 路徑
if os.getenv('APPDATA'):
    uc_path_win = os.path.join(os.getenv('APPDATA'), 'undetected_chromedriver')
    if os.path.exists(uc_path_win):
        shutil.rmtree(uc_path_win)
        print(f"   ✅ 已清除 Windows 快取: {uc_path_win}")
        cache_cleared = True

# Mac / Linux 路徑
home = os.path.expanduser("~")
uc_path_mac = os.path.join(home, '.local', 'share', 'undetected_chromedriver')
if os.path.exists(uc_path_mac):
    shutil.rmtree(uc_path_mac)
    print(f"   ✅ 已清除 Mac/Linux 快取: {uc_path_mac}")
    cache_cleared = True

if not cache_cleared:
    print("   🆗 系統中沒有殘留的快取，環境很乾淨。")

print("\n🚀 步驟二：正在嘗試重新下載並啟動 Chrome...")

try:
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1280,800")
    options.add_argument("--start-maximized")
    
    # 🌟 絕對不要加上 version_main，讓系統完全自己抓！
    driver = uc.Chrome(options=options)
    
    print("🎉 成功啦！Chrome 視窗順利彈出！")
    driver.get("https://www.google.com")
    time.sleep(3)
    driver.quit()
    print("✅ 測試完畢，你的主爬蟲程式現在可以正常運作了。")

except Exception as e:
    print("\n❌ 啟動依然失敗！請把下面這段【完整的錯誤訊息】貼給我看，我馬上幫你抓蟲：")
    print("-" * 50)
    print(e)
    print("-" * 50)

🧹 步驟一：正在尋找並清除被污染的驅動程式快取...
   ✅ 已清除 Windows 快取: C:\Users\sara9\AppData\Roaming\undetected_chromedriver

🚀 步驟二：正在嘗試重新下載並啟動 Chrome...
🎉 成功啦！Chrome 視窗順利彈出！
✅ 測試完畢，你的主爬蟲程式現在可以正常運作了。


In [7]:
import undetected_chromedriver as uc
import time

print("🚀 正在嘗試啟動全新乾淨的 Chrome 視窗...")

try:
    # 1. 移除硬編碼的版本號與固定的 user-data-dir，讓 uc 走預設初始化
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1024,768")
    options.add_argument("--start-maximized")
    
    # 2. 啟動瀏覽器 (自動適應你電腦當前的 Chrome 版本)
    driver = uc.Chrome(options=options)
    
    print("✅ 成功彈出視窗！正在導向 Google 首頁進行測試...")
    driver.get("https://www.google.com")
    
    time.sleep(5)
    print("🎉 測試完全正常，視窗即將關閉。")
    driver.quit()

except Exception as e:
    print("\n❌ 啟動失敗！捕捉到明確的錯誤訊息：")
    print(e)

🚀 正在嘗試啟動全新乾淨的 Chrome 視窗...

❌ 啟動失敗！捕捉到明確的錯誤訊息：
Message: session not created: cannot connect to chrome at 127.0.0.1:58480
from session not created: This version of ChromeDriver only supports Chrome version 147
Current browser version is 149.0.7827.103; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
	undetected_chromedriver!GetHandleVerifier [0x3bb373+10553]
	undetected_chromedriver!GetHandleVerifier [0x3bb4a4+10684]
	undetected_chromedriver!(No symbol) [0x1c1e10]
	undetected_chromedriver!(No symbol) [0x1fd452]
	undetected_chromedriver!(No symbol) [0x1fc47c]
	undetected_chromedriver!(No symbol) [0x1f2575]
	undetected_chromedriver!(No symbol) [0x1f2396]
	undetected_chromedriver!(No symbol) [0x23956f]
	undetected_chromedriver!(No symbol) [0x238d87]
	undetected_chromedriver!(No symbol) [0x22d606]
	undetected_chromedriver!(No symbol) [0x200039]
	undetected_chromedriver!(No symbol) [

In [36]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 讀取已經清洗好的「預測總表」與「特徵總表」
# ==========================================
print("📂 正在載入資料庫進行歷史回測...")
df_pred = pd.read_csv("NBA_Predicted_WS.csv")
df_features = pd.read_csv("NBA_Player_Final_Master_Features.csv")

# 清理名稱並精準合併
df_pred['Player'] = df_pred['Player'].astype(str).str.strip()
df_features['Player'] = df_features['Player'].astype(str).str.strip()

df_master = pd.merge(df_pred, df_features[['Player', 'Worst_Injury_History']], on='Player', how='left')

# 防呆：如果有人沒匹配到，預設為健康 (0分)
df_master['Worst_Injury_History'] = df_master['Worst_Injury_History'].fillna(0).astype(int)

# ==========================================
# 2. 網格搜索 (Grid Search) 尋找最佳折價權重
# ==========================================
print("⚙️ 正在載入球員真實合約成本與【專屬動態及格線】，執行 ROI 權重最佳化...")

best_weight = 0.0
best_roi_score = -9999

# 設定對應的欄位名稱
pred_col = 'Pred_Salary_Share_Percent'  # 預測理論薪資
actual_ws_col = 'Year3_WS'              # 真實第三年表現 (對答案用)
cost_col = 'Est_Cost_Percent'           # 球員的真實 CBA 規定成本
dynamic_ws_col = 'Dynamic_Truth_WS'     # 🌟 球員的專屬動態及格線 (機會成本)

# 測試從 0.0% 到 5.0% 的各種傷病折價權重 (每次微調 0.1%)
for weight in np.arange(0.0, 5.1, 0.1):
    current_roi_score = 0
    
    for index, row in df_master.iterrows():
        try:
            pred_salary = float(row[pred_col])
            actual_ws = float(row[actual_ws_col])
            injury_score = float(row['Worst_Injury_History'])
            player_real_cost = float(row[cost_col]) 
            
            # 🌟 核心修正：抓取該球員專屬的動態及格線
            player_dynamic_ws = float(row[dynamic_ws_col])
        except:
            continue # 若資料有缺漏則跳過
            
        # 💰 套用折價模型：預測薪資 - (傷病分數 * 權重)
        adjusted_salary = pred_salary - (injury_score * weight)
        
        # ⚖️ 虛擬總管決策：校正後的預測薪資，大於等於「他自己的真實成本」就執行合約
        decision_exercise = (adjusted_salary >= player_real_cost)
        
        # 🎯 判定答案：這個球員第三年真的有符合他的身價嗎？
        # 🌟 核心修正：用他專屬的 Dynamic_Truth_WS 取代僵化的 3.0
        is_good_player = (actual_ws >= player_dynamic_ws)
        
        # 🏆 ROI 評分系統 (獎懲機制)
        if decision_exercise and is_good_player:
            current_roi_score += 10  # ✅ 真陽性：慧眼識英雄，加 10 分
        elif decision_exercise and not is_good_player:
            current_roi_score -= 10  # ❌ 偽陽性：被水貨騙了，扣 10 分
        elif not decision_exercise and not is_good_player:
            current_roi_score += 5   # ✅ 真陰性：成功避開傷病水貨，加 5 分
        elif not decision_exercise and is_good_player:
            current_roi_score -= 5   # ❌ 偽陰性：防護過度錯失好球員，扣 5 分

    # 紀錄最高分的權重
    if current_roi_score > best_roi_score:
        best_roi_score = current_roi_score
        best_weight = weight

print("\n" + "="*60)
print(f"🎯 動態機會成本閉環回測完成！")
print(f"👉 根據 2000-2023 歷史資料，最佳「傷病風險折價權重」為： {best_weight:.1f} %")
print(f"📈 該權重在總管模擬決策中，獲得最高 ROI 評分： {best_roi_score} 分")
print("="*60)

📂 正在載入資料庫進行歷史回測...
⚙️ 正在載入球員真實合約成本與【專屬動態及格線】，執行 ROI 權重最佳化...

🎯 動態機會成本閉環回測完成！
👉 根據 2000-2023 歷史資料，最佳「傷病風險折價權重」為： 0.8 %
📈 該權重在總管模擬決策中，獲得最高 ROI 評分： 2725 分


In [13]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 讀取預測資料
# ==========================================
print("📂 讀取預測資料庫進行閾值回測...")
df = pd.read_csv("NBA_Predicted_WS.csv")

# 假設新秀合約成本約佔 10% (你可以替換成你真實的 Est_Cost_Percent)
if 'Est_Cost_Percent' not in df.columns:
    df['Est_Cost_Percent'] = 10.0 

# 計算理論落差 (Net Value)
df['Net_Value_Diff'] = df['Pred_Salary_Share_Percent'] - df['Est_Cost_Percent']

# ==========================================
# 2. 篩選「被財務模型淘汰」的球員
# ==========================================
# 我們只針對那些 Net_Value_Diff < 0 (理論上要被放棄) 的球員進行測試
df_declined = df[df['Net_Value_Diff'] < 0].copy()

# 確保我們有真實第三年表現來「對答案」
df_declined = df_declined.dropna(subset=['Year3_WS'])
df_declined['Year3_WS'] = pd.to_numeric(df_declined['Year3_WS'], errors='coerce')

print(f"🎯 共有 {len(df_declined)} 位球員被財務模型初步判定為【拒絕執行】。")
print("⚙️ 正在尋找最佳的「強制保留底線 (WS_Floor)」...\n")

# ==========================================
# 3. 網格搜索 (Grid Search) 尋找最佳 WS_Floor
# ==========================================
best_floor = 0.0
best_roi_score = -9999

# 測試從 0.0 到 3.0 的底線 (間隔 0.1)
for ws_floor in np.arange(0.0, 3.1, 0.1):
    current_score = 0
    
    for index, row in df_declined.iterrows():
        pred_ws = row['Pred_WS']
        actual_ws = row['Year3_WS']
        
        # 啟動安全網：如果預測 WS 大於等於我們測試的底線，強制將他「救回 (Exercise)」
        is_saved_by_net = (pred_ws >= ws_floor)
        
        # 評判標準：這個球員後來第三年真的打出輪替水準了嗎？(假設真實 WS >= 1.0 算及格)
        is_actually_useful = (actual_ws >= 1.0)
        
        # 評分機制：
        if is_saved_by_net and is_actually_useful:
            current_score += 10  # ✅ 成功救回一個可用的輪替球員！(加分)
        elif is_saved_by_net and not is_actually_useful:
            current_score -= 10  # ❌ 救錯了，他真的是個水貨，白花錢！(扣分)
        # 如果沒救他，就不加分也不扣分 (維持原本的 Decline 決策)

    # 記錄最高分
    if current_score > best_roi_score:
        best_roi_score = current_score
        best_floor = ws_floor

print("="*50)
print(f"🏆 歷史回測完成！")
print(f"👉 數據證明，最佳的『替補安全網閾值 (Replacement Level Floor)』為： Pred_WS = {best_floor:.1f}")
print("="*50)

📂 讀取預測資料庫進行閾值回測...
🎯 共有 189 位球員被財務模型初步判定為【拒絕執行】。
⚙️ 正在尋找最佳的「強制保留底線 (WS_Floor)」...

🏆 歷史回測完成！
👉 數據證明，最佳的『替補安全網閾值 (Replacement Level Floor)』為： Pred_WS = 1.2


In [16]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 讀取原始總表
# ==========================================
filename = "NBA_Normalized_Final.csv"
print(f"📂 正在讀取總表：{filename} ...")
df_main = pd.read_csv(filename)

# ==========================================
# 2. 確保有「合約成本佔比 (Est_Cost_Percent)」
# ==========================================
# 總表裡可能原本沒有這個欄位，我們根據 DraftRank 即時算出來
def get_cost_percent(row):
    try:
        rank = float(row["DraftRank"])
        if 1 <= rank <= 30:
            return 8.0 - (rank - 1) * (6.5 / 29)
    except:
        pass
    return 1.1  # 針對二輪秀或落選秀，給予預設的底薪佔比 (約1.1%)

if 'Est_Cost_Percent' not in df_main.columns:
    df_main['Est_Cost_Percent'] = df_main.apply(get_cost_percent, axis=1)

# ==========================================
# 3. 計算動態機會成本與及格線
# ==========================================
print("⚙️ 正在根據每年的經濟指標計算 Dynamic_Truth_WS ...")

TEAM_COUNT = 30
TOTAL_GAMES = 1230
VET_MIN_COST_PERCENT = 1.8 # 底薪老將約佔 1.8% 薪資空間

# 1. 算出該年度 1 WS 價值多少錢 (M)
df_main['Marginal_Value_per_WS'] = (df_main['Salary_Cap_At_Start'] * TEAM_COUNT * df_main['BRI_Factor']) / TOTAL_GAMES

# 2. 算出該年度 1 WS 佔團隊薪資空間的百分比 (%)
df_main['WS_Value_Percent'] = (df_main['Marginal_Value_per_WS'] / df_main['Salary_Cap_At_Start']) * 100

# 3. 算出該年度的底薪保底產能 (絕對下限 WS)
df_main['Absolute_Floor_WS'] = VET_MIN_COST_PERCENT / df_main['WS_Value_Percent']

# 🌟 核心寫入：計算每位球員專屬的「動態及格線」
df_main['Dynamic_Truth_WS'] = df_main['Est_Cost_Percent'] / df_main['WS_Value_Percent']

# 強制拉回底薪老將的最低產能
df_main['Dynamic_Truth_WS'] = np.maximum(df_main['Dynamic_Truth_WS'], df_main['Absolute_Floor_WS'])

# 為了讓總表保持乾淨，我們把計算過程產生的中繼欄位刪除，只留下我們要的 Dynamic_Truth_WS 和 Est_Cost_Percent
cols_to_drop = ['Marginal_Value_per_WS', 'WS_Value_Percent', 'Absolute_Floor_WS']
df_main = df_main.drop(columns=cols_to_drop)

# 將數字四捨五入到小數點後兩位，讓表格更簡潔
df_main['Dynamic_Truth_WS'] = df_main['Dynamic_Truth_WS'].round(2)
df_main['Est_Cost_Percent'] = df_main['Est_Cost_Percent'].round(2)

# ==========================================
# 4. 存回原始檔案 (覆蓋)
# ==========================================
print("💾 正在將新欄位寫回總表...")
df_main.to_csv(filename, index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(f"🎉 成功！已經將 `Dynamic_Truth_WS` 與 `Est_Cost_Percent` 永久寫入 {filename} 中。")
print("="*50)

# 印出幾筆資料檢查一下
cols_to_print = ['Player', 'year_start', 'DraftRank', 'Est_Cost_Percent', 'Dynamic_Truth_WS']
print("\n📊 隨機抽樣檢查 (首輪前五順位 vs 末段順位)：")
sample_check = pd.concat([
    df_main[df_main['DraftRank'] <= 3].head(3), 
    df_main[df_main['DraftRank'] >= 25].head(3)
])
print(sample_check[cols_to_print].to_string(index=False))

📂 正在讀取總表：NBA_Normalized_Final.csv ...
⚙️ 正在根據每年的經濟指標計算 Dynamic_Truth_WS ...
💾 正在將新欄位寫回總表...

🎉 成功！已經將 `Dynamic_Truth_WS` 與 `Est_Cost_Percent` 永久寫入 NBA_Normalized_Final.csv 中。

📊 隨機抽樣檢查 (首輪前五順位 vs 末段順位)：
         Player  year_start  DraftRank  Est_Cost_Percent  Dynamic_Truth_WS
Bill Cartwright      1980.0        3.0              7.55              2.81
  Magic Johnson      1980.0        1.0              8.00              2.98
     Jim Paxson      1980.0        3.0              7.55              2.81
      Jeff Cook      1980.0       49.0              1.10              0.67
Hollis Copeland      1980.0       46.0              1.10              0.67
  John Coughran      1980.0       74.0              1.10              0.67


In [17]:
import pandas as pd

# ==========================================
# 1. 讀取資料庫
# ==========================================
print("📂 正在讀取資料庫...")
df_pred = pd.read_csv("NBA_Predicted_WS.csv")
df_master = pd.read_csv("NBA_Normalized_Final.csv")

print(f"📊 預測表目前筆數: {df_pred.shape}")

# ==========================================
# 2. 進行資料合併 (Merge)
# ==========================================
print("🔗 正在以 [Player, year_start] 為基準將總表特徵併入...")

# 確保總表主鍵唯一，避免合併時導致預測表的列數膨脹
df_master_unique = df_master.drop_duplicates(subset=['Player', 'year_start'])

# 以預測表為主體 (left join) 合併總表特徵
df_merged = pd.merge(
    df_pred, 
    df_master_unique, 
    on=['Player', 'year_start'], 
    how='left',
    suffixes=('', '_master_dup')
)

# 自動清理因為合併而產生的重複欄位
dup_cols = [col for col in df_merged.columns if col.endswith('_master_dup')]
if dup_cols:
    df_merged = df_merged.drop(columns=dup_cols)

# ==========================================
# 3. 直接覆蓋原有的 NBA_Predicted_WS.csv
# ==========================================
output_filename = "NBA_Predicted_WS.csv"
df_merged.to_csv(output_filename, index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(f"🎉 特徵合併完成！數據已直接更新並覆蓋至：{output_filename}")
print(f"📊 更新後的大表維度：{df_merged.shape[0]} 列 x {df_merged.shape[1]} 欄")
print("="*50)

📂 正在讀取資料庫...
📊 預測表目前筆數: (710, 7)
🔗 正在以 [Player, year_start] 為基準將總表特徵併入...

🎉 特徵合併完成！數據已直接更新並覆蓋至：NBA_Predicted_WS.csv
📊 更新後的大表維度：710 列 x 71 欄


In [11]:
import os
import pandas as pd
import numpy as np

# ==========================================
# 1. 讀取現有特徵總表與原始傷病資料
# ==========================================
master_filename = "NBA_Player_Final_Master_Features.csv"
merged_filename = "merged.csv"

print("📂 正在載入資料庫...")
df_master = pd.read_csv(master_filename)
df_merged = pd.read_csv(merged_filename)

df_master['Player'] = df_master['Player'].astype(str).str.strip()
df_merged['PLAYER_NAME'] = df_merged['PLAYER_NAME'].astype(str).str.strip()

# ==========================================
# 2. 建立 DraftYear 到目標賽季的映射
# ==========================================
print("⚙️ 正在根據 DraftYear 鎖定每位球員的專屬前兩個賽季...")

def get_season_string(start_year):
    y1 = str(start_year -1)[-2:]
    y2 = str(start_year )[-2:] if (start_year + 1) % 100 != 0 else "00"
    return f"{y1}-{y2}"

def get_target_seasons(draft_year):
    if pd.isna(draft_year):
        return []
    dy = int(float(draft_year))
    # 選秀當年的球季 (例如 2010 -> '10-11') 以及下一年 ('11-12')
    # 修正：NBA選秀是在暑假，所以 2010 年新秀的第一個賽季是 '10-11'，第二個是 '11-12'
    return [get_season_string(dy), get_season_string(dy + 1)]

player_target_seasons = {}
for _, row in df_master.iterrows():
    if 'DraftYear' in row and not pd.isna(row['DraftYear']):
        player_target_seasons[row['Player']] = get_target_seasons(row['DraftYear'])

# ==========================================
# 3. 🌟 核心修正：計算 DAYS MISSED > 41 的數量
# ==========================================
print("⚙️ 正在計算每位球員前兩年中，DAYS MISSED 大於 41 的次數...")

def is_target_season(row):
    p_name = row['PLAYER_NAME']
    s_num = str(row['SEASON']).strip()
    return s_num in player_target_seasons.get(p_name, [])

# 只保留屬於該球員前兩個賽季的資料
df_merged['is_target'] = df_merged.apply(is_target_season, axis=1)
df_target = df_merged[df_merged['is_target']].copy()

# 確保 DAYS MISSED 是數值格式
df_target['DAYS MISSED'] = pd.to_numeric(df_target['DAYS MISSED'], errors='coerce')

# 判斷每次受傷是否大於 41 天
df_target['is_over_41_days'] = df_target['DAYS MISSED'] > 27

# 聚合加總：算出該球員在這兩季內，總共發生過幾次「缺賽大於41天」的嚴重傷病
severe_injury_counts = df_target.groupby('PLAYER_NAME')['is_over_41_days'].sum().to_dict()

# ==========================================
# 4. 寫回特徵總表並存檔
# ==========================================
print(f"🔗 正在將統計數量精準更新至 {master_filename}...")

# 將計算出來的次數映射到總表上，沒有資料的填入 0
df_master['Worst_Injury_History'] = df_master['Player'].map(severe_injury_counts).fillna(0).astype(int)

# 儲存覆蓋
df_master.to_csv(master_filename, index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print(f"🎉 任務成功！")
print(f"💾 已經將 DAYS MISSED > 41 的次數更新至 {master_filename} 的 Worst_Injury_History 欄位中。")
print("="*50)

# 印出最新分配狀況讓你檢查
print("\n📊 嚴重傷病次數 (Worst_Injury_History) 最新分佈：")
print(df_master['Worst_Injury_History'].value_counts())

📂 正在載入資料庫...
⚙️ 正在根據 DraftYear 鎖定每位球員的專屬前兩個賽季...
⚙️ 正在計算每位球員前兩年中，DAYS MISSED 大於 41 的次數...
🔗 正在將統計數量精準更新至 NBA_Player_Final_Master_Features.csv...

🎉 任務成功！
💾 已經將 DAYS MISSED > 41 的次數更新至 NBA_Player_Final_Master_Features.csv 的 Worst_Injury_History 欄位中。

📊 嚴重傷病次數 (Worst_Injury_History) 最新分佈：
Worst_Injury_History
0    4651
1      16
2       2
Name: count, dtype: int64


In [37]:
import pandas as pd
import os

# ==========================================
# 1. 讀取兩個檔案
# ==========================================
master_filename = "NBA_Player_Final_Master_Features.csv"  # 你的特徵總表
normalized_filename = "NBA_Normalized_Final.csv"          # 包含所有球員與年份的來源表

print(f"📂 正在讀取資料庫進行比對...")
df_master = pd.read_csv(master_filename)
df_norm = pd.read_csv(normalized_filename)

# 清洗名稱前後的空白，確保比對時 100% 精準
df_master['Player'] = df_master['Player'].astype(str).str.strip()
df_norm['Player'] = df_norm['Player'].astype(str).str.strip()

print(f"📊 比對前：特徵總表目前共有 {len(df_master)} 位球員。")

# ==========================================
# 2. 萃取標準名單與年份
# ==========================================
print("⚙️ 正在從 Normalized 總表提取球員與年份資料...")
# 抓出所有的 [姓名, 年份]，並去除重複的球員 (確保一人只有一列)
df_standard = df_norm[['Player', 'year_start']].drop_duplicates(subset=['Player']).copy()

# 將欄位名稱改為與特徵總表一致的 'DraftYear'
df_standard = df_standard.rename(columns={'year_start': 'DraftYear'})

# ==========================================
# 3. 找出缺失的球員 (差集) 並補齊
# ==========================================
print("🔍 正在進行名單交叉比對...")

# 核心邏輯：找出在標準名單中，但「不在 (~)」特徵總表中的球員
missing_players = df_standard[~df_standard['Player'].isin(df_master['Player'])].copy()

if len(missing_players) > 0:
    print(f"🚨 發現 {len(missing_players)} 位缺失的球員！正在將他們加入特徵總表...")
    
    # 將缺失的球員 (包含 Player 與 DraftYear 兩個欄位) 合併到總表下方
    df_master_updated = pd.concat([df_master, missing_players], ignore_index=True)
    
    # 存檔覆寫
    df_master_updated.to_csv(master_filename, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*50)
    print(f"🎉 補齊成功！")
    print(f"💾 檔案已更新並儲存至：{master_filename}")
    print(f"📈 更新後：特徵總表共有 {len(df_master_updated)} 位球員。")
    print("="*50)
    
    # 預覽一下到底補了哪些人
    print("\n📝 新增的球員名單預覽 (前 10 筆)：")
    print(missing_players.head(10).to_string(index=False))

else:
    print("\n" + "="*50)
    print("✅ 檢查完畢！你的特徵總表非常完整，沒有任何缺失的球員。")
    print("="*50)

📂 正在讀取資料庫進行比對...
📊 比對前：特徵總表目前共有 3326 位球員。
⚙️ 正在從 Normalized 總表提取球員與年份資料...
🔍 正在進行名單交叉比對...
🚨 發現 1343 位缺失的球員！正在將他們加入特徵總表...

🎉 補齊成功！
💾 檔案已更新並儲存至：NBA_Player_Final_Master_Features.csv
📈 更新後：特徵總表共有 4669 位球員。

📝 新增的球員名單預覽 (前 10 筆)：
         Player  DraftYear
     Larry Bird     1980.0
 Dudley Bradley     1980.0
Bill Cartwright     1980.0
      Jeff Cook     1980.0
Hollis Copeland     1980.0
  John Coughran     1980.0
   Paul Dawkins     1980.0
     Greg Deane     1980.0
    Larry Demic     1980.0
   Terry Duerod     1980.0


In [39]:
import os
import random
import time
import urllib.parse
from datetime import datetime, timezone
import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langdetect import detect, LangDetectException

# ==========================================
# --- 1. NLP 清洗與去重模組 ---
# ==========================================
def filter_english_only(news_list):
    if not news_list: return []
    english_news = []
    for news in news_list:
        text = news.get("Content", "")
        if not text.strip(): continue
        try:
            if detect(text) == 'en': english_news.append(news)
        except LangDetectException:
            continue
    return english_news

def deduplicate_news(news_list, threshold=0.85):
    if len(news_list) < 2: return news_list
    contents = [n["Content"] for n in news_list]
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(contents)
    cosine_sim = cosine_similarity(tfidf_matrix)
    
    unique_indices = []
    dropped = set()
    for i in range(len(news_list)):
        if i in dropped: continue
        unique_indices.append(i)
        for j in range(i + 1, len(news_list)):
            if cosine_sim[i, j] >= threshold:
                dropped.add(j)
    return [news_list[i] for i in unique_indices]

# ==========================================
# --- 2. 終極擬真抗封鎖抓取邏輯 ---
# ==========================================
def scrape_single_player(player_name, draft_year): 
    all_final_data = []
    
    options = uc.ChromeOptions()
    profile_path = os.path.abspath("./ultimate_year_profile")
    options.add_argument(f"--user-data-dir={profile_path}")
    options.add_argument("--window-size=1920,1080") 
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-notifications")
    options.add_argument("--lang=en-US,en;q=0.9")
    
    driver = uc.Chrome(options=options, version_main=147) # 依據你的 Chrome 版本調整
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined});"
    })
    
    # 依據選秀年份設定時間區間 (Luka 為 2018 年選秀)
    start_dt = datetime(draft_year - 1, 10, 1, tzinfo=timezone.utc)
    end_dt = datetime(draft_year + 1, 6, 30, tzinfo=timezone.utc)
    date_param = f"cdr:1,cd_min:{start_dt.strftime('%m/%d/%Y')},cd_max:{end_dt.strftime('%m/%d/%Y')}"
    query = urllib.parse.quote_plus(player_name)
    
    temp_player_news = [] 
    page = 0
    max_safe_pages = 10 
    
    print(f"\n🔎 正在檢索: {player_name} (區間: {start_dt.date()} ~ {end_dt.date()})")
    
    try:
        driver.get("https://www.google.com")
        time.sleep(random.uniform(1.5, 3.0))
        search_box = driver.find_element(By.NAME, "q")
        search_box.send_keys(player_name) 
        time.sleep(random.uniform(0.5, 1.2)) 
        search_box.send_keys(Keys.RETURN) 
        time.sleep(random.uniform(2.0, 4.0))
    except Exception:
        pass 
        
    while page < max_safe_pages:
        start_offset = page * 20
        news_url = f"https://www.google.com/search?q={query}&tbm=nws&tbs={date_param}&start={start_offset}"
        
        print(f"   ➡️ 掃描第 {page + 1} 頁...")
        driver.get(news_url)
        time.sleep(random.uniform(3.5, 6.0)) 

        try:
            scroll_depth = random.randint(400, 900)
            driver.execute_script(f"window.scrollBy(0, {scroll_depth});")
            time.sleep(random.uniform(1.0, 2.5))
        except:
            pass 

        soup = BeautifulSoup(driver.page_source, "html.parser")
        result_containers = soup.select('div[data-ved]') 
        
        if not result_containers:
            page_text = driver.page_source.lower()
            if "recaptcha" in page_text or "unusual traffic" in page_text:
                print("\n🚨 警告：觸發 Google 驗證碼！請手動解開後按 [Enter]...")
                input()
                soup = BeautifulSoup(driver.page_source, "html.parser")
                result_containers = soup.select('div[data-ved]') 
            
            if not result_containers:
                print(f"   🛑 已無新資料。")
                break
        
        page_item_count = 0
        for res in result_containers:
            title_el = res.find('div', role='heading')
            snippet_el = res.find('div', style=lambda x: x and 'line-clamp' in x)
            
            if title_el and snippet_el:
                temp_player_news.append({
                    "Player": player_name,
                    "Draft_Year": draft_year,
                    "Date": "Historical", 
                    "Content": f"{title_el.text} | {snippet_el.text}"
                })
                page_item_count += 1
        
        if page_item_count == 0: break
        page += 1 
        
    en_news = filter_english_only(temp_player_news)
    clean_news = deduplicate_news(en_news, threshold=0.85)
    print(f"✅ 爬取完畢，保留 {len(clean_news)} 筆精華數據。")
    
    driver.quit()
    return clean_news

# ==========================================
# --- 3. 執行主程式並存入指定 CSV ---
# ==========================================
if __name__ == "__main__":
    TARGET_PLAYER = "Luka Doncic"
    DRAFT_YEAR = 2018  # Luka 的選秀年份 (用來計算搜尋時間區間)
    OUTPUT_CSV = "NBA_News_2019_Full_Cohort.csv"  # 你指定的輸出檔案名稱

    # 1️⃣ 爬蟲抓取
    news_data = scrape_single_player(TARGET_PLAYER, DRAFT_YEAR)
    
    if not news_data:
        print("⚠️ 未抓取到任何資料，程式結束。")
        exit()

    # 2️⃣ 轉為 DataFrame
    df_new = pd.DataFrame(news_data)

    # 3️⃣ 安全存入指定的檔案
    print(f"\n🔄 正在將 {TARGET_PLAYER} 的新聞存入 {OUTPUT_CSV}...")
    if os.path.exists(OUTPUT_CSV):
        # 讀取舊檔案
        df_existing = pd.read_csv(OUTPUT_CSV)
        # 避免重複寫入：先把舊檔案裡如果已經有 Luka Doncic 的資料刪掉
        df_existing = df_existing[df_existing['Player'] != TARGET_PLAYER]
        # 合併新舊資料
        df_final = pd.concat([df_existing, df_new], ignore_index=True)
        print(f"   - 找到現有檔案！已將資料合併，目前檔案共 {len(df_final)} 筆新聞。")
    else:
        # 如果檔案不存在，直接當作新檔案
        df_final = df_new
        print(f"   - 檔案不存在，已自動建立新檔案，共 {len(df_final)} 筆新聞。")

    # 存檔
    df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*50)
    print(f"🎉 任務完成！{TARGET_PLAYER} 的歷史新聞已成功存入 {OUTPUT_CSV}")
    print("="*50)


🔎 正在檢索: Luka Doncic (區間: 2017-10-01 ~ 2019-06-30)
   ➡️ 掃描第 1 頁...
   ➡️ 掃描第 2 頁...
   ➡️ 掃描第 3 頁...
   ➡️ 掃描第 4 頁...
   ➡️ 掃描第 5 頁...
   ➡️ 掃描第 6 頁...
   ➡️ 掃描第 7 頁...
   ➡️ 掃描第 8 頁...
   ➡️ 掃描第 9 頁...
   ➡️ 掃描第 10 頁...
✅ 爬取完畢，保留 68 筆精華數據。

🔄 正在將 Luka Doncic 的新聞存入 NBA_News_2019_Full_Cohort.csv...
   - 找到現有檔案！已將資料合併，目前檔案共 1848 筆新聞。

🎉 任務完成！Luka Doncic 的歷史新聞已成功存入 NBA_News_2019_Full_Cohort.csv
